In [ ]:
# Stage 1: Problem Definition & Literature Review
# Diabetic Retinopathy Detection from Fundus Images

In [ ]:
print("""
PROBLEM DEFINITION — DIABETIC RETINOPATHY DETECTION
=====================================================

WHAT IS IT?
  Diabetic Retinopathy (DR) is a diabetes-induced damage to blood
  vessels in the retina. It is the leading cause of preventable
  blindness in working-age adults worldwide.

WHY DOES IT MATTER?
  • 537 million adults have diabetes globally (IDF, 2021)
  • 1 in 3 diabetic patients develops DR over their lifetime
  • Early screening reduces blindness risk by up to 94%
  • Global shortage of ophthalmologists — AI can fill this gap

OUR TASK
  Input  : Retinal fundus photograph (RGB image)
  Output : DR severity grade (0, 1, 2, 3, or 4)
  Metric : Quadratic Weighted Kappa (QWK)
""")


PROBLEM DEFINITION — DIABETIC RETINOPATHY DETECTION

WHAT IS IT?
  Diabetic Retinopathy (DR) is a diabetes-induced damage to blood
  vessels in the retina. It is the leading cause of preventable
  blindness in working-age adults worldwide.

WHY DOES IT MATTER?
  • 537 million adults have diabetes globally (IDF, 2021)
  • 1 in 3 diabetic patients develops DR over their lifetime
  • Early screening reduces blindness risk by up to 94%
  • Global shortage of ophthalmologists — AI can fill this gap

OUR TASK
  Input  : Retinal fundus photograph (RGB image)
  Output : DR severity grade (0, 1, 2, 3, or 4)
  Metric : Quadratic Weighted Kappa (QWK)



In [ ]:
import pandas as pd

grading = pd.DataFrame({
    'Grade': [0, 1, 2, 3, 4],
    'Label': ['No DR', 'Mild NPDR', 'Moderate NPDR', 'Severe NPDR', 'Proliferative DR'],
    'Clinical Signs': [
        'No lesions visible',
        'Microaneurysms only',
        'Hemorrhages, hard exudates, cotton-wool spots',
        '>20 hemorrhages/quadrant, venous beading, IRMA',
        'Neovascularization, vitreous hemorrhage'
    ],
    'Action Required': [
        'Routine screening',
        'Monitor annually',
        'Refer to specialist',
        'Urgent referral',
        'Immediate treatment'
    ]
})

print("DR SEVERITY GRADING TABLE")
print("="*80)
print(grading.to_string(index=False))
print("\n⚠  Clinical threshold: Grade ≥ 2 requires specialist referral.")
print("   Our model must minimize false negatives at this boundary.")

DR SEVERITY GRADING TABLE
 Grade            Label                                 Clinical Signs     Action Required
     0            No DR                             No lesions visible   Routine screening
     1        Mild NPDR                            Microaneurysms only    Monitor annually
     2    Moderate NPDR  Hemorrhages, hard exudates, cotton-wool spots Refer to specialist
     3      Severe NPDR >20 hemorrhages/quadrant, venous beading, IRMA     Urgent referral
     4 Proliferative DR        Neovascularization, vitreous hemorrhage Immediate treatment

⚠  Clinical threshold: Grade ≥ 2 requires specialist referral.
   Our model must minimize false negatives at this boundary.


In [ ]:
print("""
WHY QUADRATIC WEIGHTED KAPPA (QWK)?
=====================================

This is a 5-class ORDINAL classification problem.
Grades have a natural order: 0 < 1 < 2 < 3 < 4

Problem with plain accuracy:
  A model that predicts Grade 0 for every image = ~73% accuracy
  But QWK for that model = 0.0  (completely useless clinically)

How QWK works:
  - Penalizes predictions proportional to how far off they are
  - Predicting Grade 0 when truth is Grade 4 → heavy penalty
  - Predicting Grade 1 when truth is Grade 2 → small penalty

QWK Score Guide:
  < 0.20   →  Slight agreement
  0.21-0.40 → Fair
  0.41-0.60 → Moderate
  0.61-0.80 → Substantial
  0.81-1.00 → Almost perfect  ← Our target

Target: QWK ≥ 0.80
Kaggle competition top solutions: 0.84 – 0.86
""")


WHY QUADRATIC WEIGHTED KAPPA (QWK)?

This is a 5-class ORDINAL classification problem.
Grades have a natural order: 0 < 1 < 2 < 3 < 4

Problem with plain accuracy:
  A model that predicts Grade 0 for every image = ~73% accuracy
  But QWK for that model = 0.0  (completely useless clinically)

How QWK works:
  - Penalizes predictions proportional to how far off they are
  - Predicting Grade 0 when truth is Grade 4 → heavy penalty
  - Predicting Grade 1 when truth is Grade 2 → small penalty

QWK Score Guide:
  < 0.20   →  Slight agreement
  0.21-0.40 → Fair
  0.41-0.60 → Moderate
  0.61-0.80 → Substantial
  0.81-1.00 → Almost perfect  ← Our target

Target: QWK ≥ 0.80
Kaggle competition top solutions: 0.84 – 0.86



In [ ]:
papers = [
    {
        "id": "[1]",
        "citation": "Gulshan et al. (2016) — Nature",
        "title": "Development and Validation of a Deep Learning Algorithm for DR",
        "relevance": "First landmark DL paper for DR. 128k images, AUC=0.99. Proves DL can match ophthalmologist performance. Validates our approach."
    },
    {
        "id": "[2]",
        "citation": "He et al. (2016) — CVPR",
        "title": "Deep Residual Learning for Image Recognition (ResNet)",
        "relevance": "Introduced skip connections solving vanishing gradients. ResNet-50 (25.6M params) is our baseline backbone, pretrained on ImageNet."
    },
    {
        "id": "[3]",
        "citation": "Tan & Le (2019) — ICML",
        "title": "EfficientNet: Rethinking Model Scaling for CNNs",
        "relevance": "Compound scaling of depth/width/resolution. EfficientNet-B4 (19.3M params) is our main model — better accuracy with fewer parameters."
    },
    {
        "id": "[4]",
        "citation": "Selvaraju et al. (2017) — ICCV",
        "title": "Grad-CAM: Visual Explanations from Deep Networks",
        "relevance": "Gradient-based class activation heatmaps. Critical for clinical trust — doctors need to see what the model focuses on."
    },
    {
        "id": "[5]",
        "citation": "Graham (2015) — Kaggle DR Competition Winner",
        "title": "Kaggle Diabetic Retinopathy Detection: 1st Place Solution",
        "relevance": "Local average subtraction + circular crop preprocessing. Domain-specific trick that significantly improves feature visibility."
    },
]

print("LITERATURE REVIEW")
print("="*70)
for p in papers:
    print(f"\n{p['id']} {p['citation']}")
    print(f"   Title     : {p['title']}")
    print(f"   Relevance : {p['relevance']}")
print()

LITERATURE REVIEW

[1] Gulshan et al. (2016) — Nature
   Title     : Development and Validation of a Deep Learning Algorithm for DR
   Relevance : First landmark DL paper for DR. 128k images, AUC=0.99. Proves DL can match ophthalmologist performance. Validates our approach.

[2] He et al. (2016) — CVPR
   Title     : Deep Residual Learning for Image Recognition (ResNet)
   Relevance : Introduced skip connections solving vanishing gradients. ResNet-50 (25.6M params) is our baseline backbone, pretrained on ImageNet.

[3] Tan & Le (2019) — ICML
   Title     : EfficientNet: Rethinking Model Scaling for CNNs
   Relevance : Compound scaling of depth/width/resolution. EfficientNet-B4 (19.3M params) is our main model — better accuracy with fewer parameters.

[4] Selvaraju et al. (2017) — ICCV
   Title     : Grad-CAM: Visual Explanations from Deep Networks
   Relevance : Gradient-based class activation heatmaps. Critical for clinical trust — doctors need to see what the model focuses on.

[5] G

In [ ]:
print("""
OUR APPROACH — FULL PIPELINE
==============================

  Fundus Image (RGB, 224×224)
         │
         ▼
  ┌─────────────────────────────┐
  │  Ben Graham Preprocessing   │  ← removes dark borders,
  │  (local contrast boost)     │    enhances vessels/lesions
  └─────────────────────────────┘
         │
         ▼
  ┌─────────────────────────────┐
  │  Albumentations Augmentation│  ← flips, rotation, brightness,
  │  (train only)               │    HSV jitter, CoarseDropout
  └─────────────────────────────┘
         │
         ▼
  ┌─────────────────────────────┐
  │  Transfer Learning Backbone │  ← ResNet-50 or EfficientNet-B4
  │  (ImageNet pretrained)      │    (pretrained weights, fine-tuned)
  └─────────────────────────────┘
         │
         ▼
  ┌─────────────────────────────┐
  │  Custom Classification Head │  ← Dropout→FC(512)→ReLU→
  │                             │    Dropout→FC(5)
  └─────────────────────────────┘
         │
         ▼
  ┌─────────────────────────────┐
  │  Weighted CrossEntropy Loss │  ← handles class imbalance
  │  + WeightedRandomSampler    │    (73% Grade 0 problem)
  └─────────────────────────────┘
         │
         ▼
  ┌─────────────────────────────┐
  │  AdamW + CosineAnnealingLR  │  ← optimizer + scheduler
  └─────────────────────────────┘
         │
         ▼
  ┌─────────────────────────────┐
  │  Evaluation: QWK, Accuracy  │
  │  Confusion Matrix           │
  └─────────────────────────────┘
         │
         ▼
  ┌─────────────────────────────┐
  │  Grad-CAM Explainability    │  ← heatmaps on retinal features
  └─────────────────────────────┘
         │
         ▼
  ┌─────────────────────────────┐
  │  Streamlit Deployment       │  ← live web app, public URL
  └─────────────────────────────┘

Key Challenges Addressed:
  ✅ Class imbalance (73:1 ratio)    → weighted loss + oversampling
  ✅ Subtle inter-grade differences  → transfer learning + fine-tuning
  ✅ Image quality variability       → Ben Graham preprocessing
  ✅ Clinical trust / explainability → Grad-CAM heatmaps
  ✅ Ordinal label structure         → QWK as primary metric
""")


OUR APPROACH — FULL PIPELINE

  Fundus Image (RGB, 224×224)
         │
         ▼
  ┌─────────────────────────────┐
  │  Ben Graham Preprocessing   │  ← removes dark borders,
  │  (local contrast boost)     │    enhances vessels/lesions
  └─────────────────────────────┘
         │
         ▼
  ┌─────────────────────────────┐
  │  Albumentations Augmentation│  ← flips, rotation, brightness,
  │  (train only)               │    HSV jitter, CoarseDropout
  └─────────────────────────────┘
         │
         ▼
  ┌─────────────────────────────┐
  │  Transfer Learning Backbone │  ← ResNet-50 or EfficientNet-B4
  │  (ImageNet pretrained)      │    (pretrained weights, fine-tuned)
  └─────────────────────────────┘
         │
         ▼
  ┌─────────────────────────────┐
  │  Custom Classification Head │  ← Dropout→FC(512)→ReLU→
  │                             │    Dropout→FC(5)
  └─────────────────────────────┘
         │
         ▼
  ┌─────────────────────────────┐
  │  Weighted CrossEntropy 